# Santa 2025 - Christmas Tree Packing Challenge

This notebook addresses the Santa 2025 Kaggle competition, where the goal is to pack a variable number of Christmas trees (from 1 to 200) into the smallest possible square box.

## Challenge Overview
- **Objective**: Minimize the side length $s$ of a square bounding box containing $n$ trees.
- **Metric**: Sum of normalized areas: $\sum (s^2 / n)$.
- **Constraints**: Trees must not overlap.
- **Output**: A CSV file with `id`, `x`, `y`, `deg` for each tree in each configuration.

## 1. Import Required Libraries
We will use `shapely` for geometric operations (polygons, rotation, translation, intersection checks), `matplotlib` for visualization, and standard libraries for data handling.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, Point
from shapely import affinity
from decimal import Decimal
import math
import random

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

## 2. Define Christmas Tree Geometry

We use the `ChristmasTree` class definition provided in the sample code. This class constructs a `shapely.geometry.Polygon` representing the tree based on fixed dimensions.

**Tree Dimensions Analysis:**
- **Tip Y**: 0.8
- **Trunk Bottom Y**: -0.2
- **Total Height**: $0.8 - (-0.2) = 1.0$
- **Base Width**: 0.7
- **Max Width**: 0.7

The tree is centered relative to `(center_x, center_y)` and rotated by `angle`. Note that the competition requires high precision, so we use `Decimal` for initial construction, though `shapely` uses floats internally.

In [ ]:
scale_factor = Decimal('1.0')

class ChristmasTree:
    """Represents a single, rotatable Christmas tree of a fixed size."""

    def __init__(self, center_x='0', center_y='0', angle='0'):
        """Initializes the Christmas tree with a specific position and rotation."""
        self.center_x = Decimal(center_x)
        self.center_y = Decimal(center_y)
        self.angle = Decimal(angle)

        trunk_w = Decimal('0.15')
        trunk_h = Decimal('0.2')
        base_w = Decimal('0.7')
        mid_w = Decimal('0.4')
        top_w = Decimal('0.25')
        tip_y = Decimal('0.8')
        tier_1_y = Decimal('0.5')
        tier_2_y = Decimal('0.25')
        base_y = Decimal('0.0')
        trunk_bottom_y = -trunk_h

        initial_polygon = Polygon(
            [
                # Start at Tip
                (Decimal('0.0') * scale_factor, tip_y * scale_factor),
                # Right side - Top Tier
                (top_w / Decimal('2') * scale_factor, tier_1_y * scale_factor),
                (top_w / Decimal('4') * scale_factor, tier_1_y * scale_factor),
                # Right side - Middle Tier
                (mid_w / Decimal('2') * scale_factor, tier_2_y * scale_factor),
                (mid_w / Decimal('4') * scale_factor, tier_2_y * scale_factor),
                # Right side - Bottom Tier
                (base_w / Decimal('2') * scale_factor, base_y * scale_factor),
                # Right Trunk
                (trunk_w / Decimal('2') * scale_factor, base_y * scale_factor),
                (trunk_w / Decimal('2') * scale_factor, trunk_bottom_y * scale_factor),
                # Left Trunk
                (-(trunk_w / Decimal('2')) * scale_factor, trunk_bottom_y * scale_factor),
                (-(trunk_w / Decimal('2')) * scale_factor, base_y * scale_factor),
                # Left side - Bottom Tier
                (-(base_w / Decimal('2')) * scale_factor, base_y * scale_factor),
                # Left side - Middle Tier
                (-(mid_w / Decimal('4')) * scale_factor, tier_2_y * scale_factor),
                (-(mid_w / Decimal('2')) * scale_factor, tier_2_y * scale_factor),
                # Left side - Top Tier
                (-(top_w / Decimal('4')) * scale_factor, tier_1_y * scale_factor),
                (-(top_w / Decimal('2')) * scale_factor, tier_1_y * scale_factor),
            ]
        )
        # Rotate first, then translate
        rotated = affinity.rotate(initial_polygon, float(self.angle), origin=(0, 0))
        self.polygon = affinity.translate(rotated,
                                          xoff=float(self.center_x * scale_factor),
                                          yoff=float(self.center_y * scale_factor))

# Test creating a tree
tree = ChristmasTree(0, 0, 0)
print(f"Tree Area: {tree.polygon.area}")
print(f"Tree Bounds: {tree.polygon.bounds}")

In [ ]:
def plot_packing(trees, title="Christmas Tree Packing"):
    fig, ax = plt.subplots(figsize=(10, 10))
    
    for tree in trees:
        x, y = tree.polygon.exterior.xy
        ax.fill(x, y, alpha=0.5, fc='green', ec='black')
        
    ax.set_aspect('equal')
    ax.set_title(title)
    plt.grid(True)
    plt.show()

# Visualize a single tree
plot_packing([ChristmasTree(0, 0, 0)], "Single Tree Visualization")

In [ ]:
def simple_grid_packing(num_trees, spacing=1.0):
    trees = []
    cols = int(np.ceil(np.sqrt(num_trees)))
    
    for i in range(num_trees):
        row = i // cols
        col = i % cols
        
        x = col * spacing
        y = row * spacing
        
        # Create a tree at this position with 0 rotation
        tree = ChristmasTree(x, y, 0)
        trees.append(tree)
        
    return trees

# Generate a sample packing of 25 trees
packed_trees = simple_grid_packing(25, spacing=0.8) # Spacing 0.8 based on max width 0.7 + buffer
plot_packing(packed_trees, "Grid Packing (25 Trees)")

In [ ]:
def check_overlaps(trees):
    overlaps = 0
    for i in range(len(trees)):
        for j in range(i + 1, len(trees)):
            if trees[i].polygon.intersects(trees[j].polygon):
                overlaps += 1
                # Optional: Print overlapping pairs
                # print(f"Overlap detected between Tree {i} and Tree {j}")
    return overlaps

num_overlaps = check_overlaps(packed_trees)
print(f"Total Overlaps: {num_overlaps}")

if num_overlaps == 0:
    print("Packing is VALID.")
else:
    print("Packing is INVALID.")

In [ ]:
import pandas as pd

def save_submission(trees, filename="submission.csv"):
    data = []
    for i, tree in enumerate(trees):
        data.append({
            'id': i,
            'x': float(tree.center_x),
            'y': float(tree.center_y),
            'angle': float(tree.angle)
        })
    
    df = pd.DataFrame(data)
    df.to_csv(filename, index=False)
    print(f"Submission saved to {filename}")
    return df

# Save our sample packing
submission_df = save_submission(packed_trees)
submission_df.head()

## 6. Generate Submission

Save the tree configurations to a CSV file.

## 5. Evaluation

We need to check if the packing is valid (no overlaps).
We can use `shapely`'s `intersects` method.

## 4. Optimization Strategy (Baseline)

We will implement a simple grid-based packing strategy as a baseline. This places trees in a regular grid pattern.
For the actual competition, you would replace this with a more sophisticated algorithm (e.g., Simulated Annealing, Genetic Algorithms, or Physics-based simulation).

## 3. Visualization Helper Functions

We need a way to visualize our packing to ensure trees are not overlapping and are contained within the bounds.